In [1]:
import cv2
import pandas as pd
import supervision as sv
from ultralytics import YOLO
from trackers import SORTTracker

# Detection

MegaDetector is an open-source AI model developed by Microsoft to automatically detect animals, humans, and vehicles in camera trap images. It is built directly on the YOLO object detection architecture, leveraging its real-time processing capabilities to filter out empty frames and speed up wildlife monitoring.

For tracking details see the following reference: https://trackers.roboflow.com/latest/trackers/sort/#run-on-video-webcam-or-rtsp-stream

In [2]:
# Load model.
model_path = "md_v1000.0.0-sorrel.pt"
model_detection = YOLO(model=model_path)

In [3]:
# Get input data.
video_path = "videos/casanova/example2.mp4"
output_video_path = "output.mp4"

In [4]:
# Ignore some detections.
confidence_threshold = 0.8

In [5]:
# Open input video.
video = cv2.VideoCapture(video_path)
# Get video properties.
fps = int(video.get(cv2.CAP_PROP_FPS))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
# Create output video containing motion detection and related predictions.
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
output_video = cv2.VideoWriter(output_video_path, fourcc, fps=fps, frameSize=(width, height))

# Initialize tracker.
tracker = SORTTracker()
# Initialize visualization tools.
box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK)

tracking_data = []
frame_id = 0
while video.isOpened():
    success, frame = video.read()
    
    if not success:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detection.
    results = model_detection(frame_rgb, device="cuda", verbose=False)
    result = results[0]

    # Tracking.
    if len(result.boxes) > 0:
        detections = sv.Detections(
            xyxy=result.boxes.xyxy.cpu().numpy(),
            confidence=result.boxes.conf.cpu().numpy(),
            class_id=result.boxes.cls.cpu().numpy().astype(int),
        )
        detections = detections[detections.confidence >= confidence_threshold]
    else:
        detections = sv.Detections.empty()
        
    detections = tracker.update(detections)

    # Store detection and tracking results.
    for xyxy_box, confidence, class_id, tracker_id in zip(
            detections.xyxy,
            detections.confidence,
            detections.class_id,
            detections.tracker_id,
        ):
            tracking_data.append(
                {
                    "frame_id": frame_id,
                    "tracker_id": tracker_id,
                    "xmin": xyxy_box[0],
                    "ymin": xyxy_box[1],
                    "xmax": xyxy_box[2],
                    "ymax": xyxy_box[3],
                    "confidence": confidence,
                    "class_id": class_id,
                }
            )

    # Visualization.
    labels = [f"#{tracker_id}" for tracker_id in detections.tracker_id]
    frame = box_annotator.annotate(scene=frame, detections=detections)
    frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)

    output_video.write(frame)

    frame_id += 1

video.release()
output_video.release()
cv2.destroyAllWindows()

In [6]:
df = pd.DataFrame(tracking_data)
df

,frame_id,tracker_id,xmin,ymin,xmax,ymax,confidence,class_id
0,0,-1,1152.952515,0.000000,1279.969116,713.389832,0.920573,1
1,1,-1,1152.942871,0.000000,1279.970703,713.303101,0.920202,1
2,2,0,1159.229614,0.000000,1279.966431,712.173340,0.916315,1
3,3,0,1163.907593,0.000000,1280.000000,711.093872,0.910939,1
4,4,0,1165.143433,0.040202,1280.000000,708.592468,0.912519,1
...,...,...,...,...,...,...,...,...
347,262,1,429.638031,355.155701,507.577484,510.519867,0.850276,1
348,262,2,566.346436,410.704437,600.852966,507.298523,0.816254,1
349,263,2,567.093018,410.505066,600.594666,507.091003,0.823916,1
350,264,1,440.561371,353.001099,505.827972,506.979309,0.836961,1


In [7]:
df["tracker_id"].unique()

array([-1,  0,  1,  2])